<a href="https://colab.research.google.com/github/AmnaTariqRana/Transformers-RAG/blob/main/nlp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import json
import gzip
import os
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

In [ ]:
def load_reviews(filepath, category_name, sample_size=12000):
    records = []

    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line.strip())
                text = obj.get('reviewText', '').strip()
                rating = obj.get('overall', None)

                if text and rating is not None:
                    records.append({
                        'reviewText': text,
                        'overall': int(rating),
                        'category': category_name
                    })
            except Exception:
                continue

    df = pd.DataFrame(records)


    if len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)

    print(f"{category_name}: loaded {len(df)} reviews")
    return df

In [ ]:
BASE_PATH = "/content/data"
beauty_df     = load_reviews(os.path.join(BASE_PATH, "beauty.json.gz"),      "Beauty",      sample_size=12000)
cellphones_df = load_reviews(os.path.join(BASE_PATH, "cellphones.json.gz"),  "Cellphones",  sample_size=12000)
home_df       = load_reviews(os.path.join(BASE_PATH, "home.json.gz"),        "Home",        sample_size=12000)

Beauty: loaded 12000 reviews
Cellphones: loaded 12000 reviews
Home: loaded 12000 reviews


In [ ]:
df = pd.concat([beauty_df, cellphones_df, home_df], ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


print(f"\nTotal samples: {len(df)}")
print(f"Rating distribution:\n{df['overall'].value_counts().sort_index()}")
print(f"Category distribution:\n{df['category'].value_counts()}")
print(f"Null check:\n{df.isnull().sum()}")
print(f"\nSample row:\n{df.iloc[0]}")


Total samples: 36000
Rating distribution:
overall
1     1995
2     1933
3     3666
4     7138
5    21268
Name: count, dtype: int64
Category distribution:
category
Cellphones    12000
Home          12000
Beauty        12000
Name: count, dtype: int64
Null check:
reviewText    0
overall       0
category      0
dtype: int64

Sample row:
reviewText    I loved this product, it's really pretty and v...
overall                                                       5
category                                             Cellphones
Name: 0, dtype: object


#  Map ratings to sentiment labels

In [ ]:
def map_sentiment(rating):
    if rating <= 2:
        return 0   # neg
    elif rating == 3:
        return 1   # neutral
    else:
        return 2   # pos

df['sentiment'] = df['overall'].apply(map_sentiment)

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
print("Sentiment distribution:")
print(df['sentiment'].map(label_names).value_counts())

Sentiment distribution:
sentiment
Positive    28406
Negative     3928
Neutral      3666
Name: count, dtype: int64


#  Train/Val/Test split (70/15/15)

In [ ]:
#first split off 70% train
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['sentiment'])

#split remaining 30% into 15/15
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['sentiment'])

print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print(f"Test size  : {len(test_df)}")

for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name} sentiment distribution:")
    print(split['sentiment'].map(label_names).value_counts())

Train size : 25200
Val size   : 5400
Test size  : 5400

Train sentiment distribution:
sentiment
Positive    19884
Negative     2750
Neutral      2566
Name: count, dtype: int64

Val sentiment distribution:
sentiment
Positive    4261
Negative     589
Neutral      550
Name: count, dtype: int64

Test sentiment distribution:
sentiment
Positive    4261
Negative     589
Neutral      550
Name: count, dtype: int64


#  Save splits

In [ ]:
os.makedirs("results", exist_ok=True)

train_df.to_csv("results/train.csv", index=False)
val_df.to_csv("results/val.csv",   index=False)
test_df.to_csv("results/test.csv",  index=False)

print("Splits saved to results/")

Splits saved to results/


#  Text Cleaning

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)           # remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)        # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()     # collapse spaces
    return text

train_df['clean_text'] = train_df['reviewText'].apply(clean_text)
val_df['clean_text']   = val_df['reviewText'].apply(clean_text)
test_df['clean_text']  = test_df['reviewText'].apply(clean_text)

print("Sample cleaned text:")
print(train_df['clean_text'].iloc[0])

Sample cleaned text:
when i straighten my hair i sometimes have little hairs sticking up by my bangs so i put some of this in my hands and rubbed my hands together and gently go over the top of my hair and all of the stray hairs stay in place all day


# Tokenization

In [ ]:
def tokenize(text): #white space tokenizer
    return text.split()

train_df['tokens'] = train_df['clean_text'].apply(tokenize)
val_df['tokens']   = val_df['clean_text'].apply(tokenize)
test_df['tokens']  = test_df['clean_text'].apply(tokenize)

print("Sample tokens:")
print(train_df['tokens'].iloc[0])

Sample tokens:
['when', 'i', 'straighten', 'my', 'hair', 'i', 'sometimes', 'have', 'little', 'hairs', 'sticking', 'up', 'by', 'my', 'bangs', 'so', 'i', 'put', 'some', 'of', 'this', 'in', 'my', 'hands', 'and', 'rubbed', 'my', 'hands', 'together', 'and', 'gently', 'go', 'over', 'the', 'top', 'of', 'my', 'hair', 'and', 'all', 'of', 'the', 'stray', 'hairs', 'stay', 'in', 'place', 'all', 'day']


# Vocab Construction

In [ ]:
from collections import Counter

MIN_FREQ = 2

token_counts = Counter()
for tokens in train_df['tokens']:
    token_counts.update(tokens)

print(f"Total unique tokens before filtering: {len(token_counts)}")

#filter by min freq
vocab_tokens = [word for word, count in token_counts.items() if count >= MIN_FREQ]
print(f"Vocab size after filtering (min_freq={MIN_FREQ}): {len(vocab_tokens)}")

#special tokens
# <PAD> : padding token (index 0)
# <UNK> : unknown token for words not in vocab (index 1)
# <SOS> : start of sequence needed for decoder in Part C (index 2)
# <EOS> : end of sequence needed for decoder in Part C (index 3)

special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>']

vocab = special_tokens + sorted(vocab_tokens)  # sorted for reprod

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX = word2idx['<PAD>']   # 0
UNK_IDX = word2idx['<UNK>']   # 1
SOS_IDX = word2idx['<SOS>']   # 2
EOS_IDX = word2idx['<EOS>']   # 3

print(f"\nFinal vocab size: {VOCAB_SIZE}")
print(f"PAD={PAD_IDX}, UNK={UNK_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}")

Total unique tokens before filtering: 32547
Vocab size after filtering (min_freq=2): 18544

Final vocab size: 18548
PAD=0, UNK=1, SOS=2, EOS=3


# Cell 11 - Convert Tokens to Indices

In [ ]:
MAX_SEQ_LEN = 128

def tokens_to_indices(tokens, word2idx, max_len):
    indices = [word2idx.get(tok, UNK_IDX) for tok in tokens[:max_len]]


    padding = [PAD_IDX] * (max_len - len(indices))
    indices = indices + padding

    return indices

train_df['input_ids'] = train_df['tokens'].apply(
    lambda t: tokens_to_indices(t, word2idx, MAX_SEQ_LEN))
val_df['input_ids']   = val_df['tokens'].apply(
    lambda t: tokens_to_indices(t, word2idx, MAX_SEQ_LEN))
test_df['input_ids']  = test_df['tokens'].apply(
    lambda t: tokens_to_indices(t, word2idx, MAX_SEQ_LEN))

print("Sample input_ids (first 20 tokens):")
print(train_df['input_ids'].iloc[0][:20])
print(f"Length check: {len(train_df['input_ids'].iloc[0])}")

Sample input_ids (first 20 tokens):
[18065, 7832, 15598, 10390, 7214, 7832, 15041, 7365, 9275, 7231, 15520, 17416, 2142, 10390, 1139, 14952, 7832, 12635, 15032, 10883]
Length check: 128


# Cell 12 - Save vocab and preprocessed data

In [ ]:
import pickle

os.makedirs("models", exist_ok=True)

with open("models/vocab.pkl", "wb") as f:
    pickle.dump({
        'word2idx': word2idx,
        'idx2word': idx2word,
        'vocab_size': VOCAB_SIZE,
        'pad_idx': PAD_IDX,
        'unk_idx': UNK_IDX,
        'sos_idx': SOS_IDX,
        'eos_idx': EOS_IDX
    }, f)


train_df.to_csv("results/train_preprocessed.csv", index=False)
val_df.to_csv("results/val_preprocessed.csv",     index=False)
test_df.to_csv("results/test_preprocessed.csv",   index=False)

print(f"Vocab saved to models/vocab.pkl")
print(f"Preprocessed splits saved to results/")

Vocab saved to models/vocab.pkl
Preprocessed splits saved to results/


# Cell 13 -  stats for report

In [ ]:
#token length distribution
all_lengths = train_df['tokens'].apply(len)

print("Token length statistics (training set):")
print(f"  Mean   : {all_lengths.mean():.1f}")
print(f"  Median : {all_lengths.median():.1f}")
print(f"  90th % : {all_lengths.quantile(0.90):.1f}")
print(f"  95th % : {all_lengths.quantile(0.95):.1f}")
print(f"  Max    : {all_lengths.max()}")

coverage = (all_lengths <= MAX_SEQ_LEN).mean() * 100
print(f"\nReviews fully covered by MAX_SEQ_LEN={MAX_SEQ_LEN}: {coverage:.1f}%")

Token length statistics (training set):
  Mean   : 96.0
  Median : 57.0
  90th % : 209.0
  95th % : 294.0
  Max    : 4063

Reviews fully covered by MAX_SEQ_LEN=128: 78.8%
